## 🔐 Handling Secrets & API Keys in Colab

Since this project deals with cybersecurity, you may need to use sensitive data  
(such as Cloud API keys or private signing keys) **without hardcoding them in your script**.

**Google Colab** provides a secure **Secrets Manager** for storing such values safely.

---

### 📝 How to Use It:

1. Click on the **key icon (🔑)** in the left sidebar.
2. Add a new secret:
   - **Name:** `VIF_API_KEY` *(example)*
   - **Value:** `your_secret_key_here`
3. Toggle **Notebook access** to **ON** so the notebook can read the secret.

---

### 📌 Accessing the Secret in Code:

```python
import os
vif_key = os.environ["VIF_API_KEY"]
print("Key loaded successfully!")


In [ ]:
# @title 🚀 Step 1: Clone Repository & Install Dependencies
# @markdown Click the play button to clone the VIF engine from GitHub and set up the environment.

import os

# 1. Clone the Repository
if not os.path.exists("VIF"):
    print("🔄 Cloning VIF Repository from GitHub...")
    !git clone https://github.com/Ahmed-alrashidi/VIF.git
    %cd VIF
else:
    print("✅ Repository already cloned. Pulling latest changes...")
    %cd VIF
    !git pull

# 2. Install System Dependencies (Linux)
print("\n🛠️ Installing System Libraries (CairoSVG & Tesseract)...")
!apt-get update -qq
!apt-get install -y libcairo2 fonts-liberation tesseract-ocr -qq

# 3. Install Python Requirements
print("\n📦 Installing Python Libraries...")
!pip install -r requirements.txt

print("\n✅ Environment Ready. Project loaded in 'VIF' folder.")

In [ ]:
# @title 📤 Step 2: Upload Assets (Logo & Document)
# @markdown 1. Upload your **LOGO.svg**.
# @markdown 2. Upload the **Document** (PDF/JPG) to protect.

from google.colab import files
import os
import shutil

# Return to repo dir if needed
if os.getcwd().split('/')[-1] != 'VIF':
    os.chdir('/content/VIF')

print("📂 Please upload 'LOGO.svg' (Vector Logo):")
uploaded_logo = files.upload()

print("\n📄 Now upload your Target Document (PDF or JPG):")
uploaded_doc = files.upload()

# Handle Logo Filename
logo_target = "LOGO.svg"
found_logo = False
for filename in uploaded_logo.keys():
    if filename.lower().endswith('.svg'):
        # Rename to standard name expected by VIF_Generator
        os.rename(filename, logo_target)
        found_logo = True
        print(f"✅ Logo configured: {logo_target}")
        break

if not found_logo:
    print("⚠️ Warning: No SVG file found. VIF Generator requires a .svg file.")

# Handle Document Filename
doc_filename = None
if uploaded_doc:
    doc_filename = next(iter(uploaded_doc))
    print(f"✅ Target Document: {doc_filename}")

In [ ]:
# @title 🛡️ Step 3: Run VIF Engine (Generate -> Overlay -> Scan)
# @markdown This cell imports the logic directly from your GitHub code.

import sys
# Add current directory to python path to import modules
sys.path.append('/content/VIF')

# Import your modules from the cloned repo
try:
    from VIF_Generator import VisualFingerprintGenerator
    from VIF_Overlay import protect_document
    from VIF_Forensic_Scanner import analyze_document_advanced
    print("✅ VIF Modules Imported Successfully.")
except ImportError as e:
    print(f"❌ Error Importing Modules: {e}. Make sure you ran Step 1.")

# --- EXECUTION FLOW ---
if 'VisualFingerprintGenerator' in locals() and os.path.exists("LOGO.svg") and doc_filename:

    # 1. GENERATE
    print("\n⚙️ [1/3] Generating Fingerprint...")
    # Read SVG data
    with open("LOGO.svg", "rb") as f: svg_data = f.read()

    # Initialize & Create
    gen = VisualFingerprintGenerator()
    # You can customize User/Device here or use defaults
    fp_img, _ = gen.create_fingerprint(svg_data, "109207", "WS-RIY-01")

    # Save the high-res image for the overlay module
    fp_path = "fingerprint_highres.png"
    fp_img.save(fp_path)
    print("   -> Fingerprint created.")

    # 2. PROTECT (Overlay)
    print("\n🔒 [2/3] Protecting Document...")
    # Note: VIF_Overlay.py usually expects the file on disk.
    # We call the function directly if modified to accept args,
    # OR we simulate the main execution.
    # Assuming protect_document takes (input_file) and looks for fingerprint_highres.png

    # Let's ensure the overlay module finds the fingerprint
    if os.path.exists(fp_path):
        # We need to adapt this call based on your exact VIF_Overlay.py structure
        # If your script has a protect_document(file) function:
        try:
             # Based on V1.0 structure
             # We might need to make sure VIF_Overlay uses the local fingerprint
             protected_path = f"Protected_{doc_filename}"
             if not protected_path.endswith('.pdf') and '.' in protected_path:
                 protected_path = os.path.splitext(protected_path)[0] + ".pdf"

             # Call the protection logic (assuming function availability)
             # If strictly command line:
             !python VIF_Overlay.py "$doc_filename"

             # Check if output exists
             if os.path.exists(protected_path):
                 print(f"   -> Secure document saved: {protected_path}")

                 # 3. ANALYZE (Scanner)
                 print("\n🕵️ [3/3] Running Forensic Scan...")
                 # Call Scanner via command line for robust output
                 !python VIF_Forensic_Scanner.py "$protected_path"

                 print("\n✅ DEMO COMPLETE.")

                 # Download Button
                 from google.colab import files
                 files.download(protected_path)
             else:
                 print("❌ Error: Protected file was not created.")

        except Exception as e:
            print(f"❌ Execution Error: {e}")

else:
    print("❌ Critical: Missing Logo or Document. Please check Step 2.")